# RAG-Powered Method Cards using Structured Knowledge

**A Technical Deep Dive into Intelligent Statistical & ML Method Retrieval**

---

### Key Innovation

Instead of splitting documentation into arbitrary text chunks, we created **40+ curated method cards** - structured knowledge units containing:
- Decision criteria (when to use / when not to use)
- Data requirements (sample size, normality, missing values)
- Code examples and parameters
- Alternatives and interpretations

This enables **constraint-aware retrieval** that filters methods based on actual data characteristics.

## 1. The Method Card Solution

### What is a Method Card?

A **method card** is a structured knowledge unit that contains everything needed to make an informed decision about using a statistical test, ML algorithm, or preprocessing technique.

**Structure:**

```yaml
method_name: "SimpleImputer (median)"
category: "preprocessing_imputation"
problem_type: "missing_values"

# Data requirements and constraints
data_conditions:
  sample_size_min: 10
  handles_missing: true
  handles_outliers: true
  handles_non_normal: true

# Decision logic
when_to_use: "Numeric features with missing values. Skewed distributions. 
              Outliers present. More robust than mean for non-normal data."
when_not_to_use: "Categorical features. Missing > 40%. Time series with trend."

# Alternatives and code
alternatives: ["SimpleImputer (mean)", "KNNImputer"]
code_example: |
  from sklearn.impute import SimpleImputer
  imputer = SimpleImputer(strategy='median')
  X_imputed = imputer.fit_transform(X)
```

### Key Advantages

* **Complete decision unit**: All information in one place  
* **Constraint-aware**: Can filter by data characteristics  
* **Curated content**: High-quality, actionable guidance  
* **Easy to extend**: Add new methods via YAML  
* **Version controlled**: Track changes to recommendations

## 2. Architecture Overview

### System Components

```
┌─────────────────────────────────────────────────────────┐
│                    User Query                           │
│        "impute missing values in dataset"               │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│              RAG System (rag_system.py)                 │
│  • Semantic search query embedding                      │
│  • Metadata filtering (topic, category)                 │
│  • Data profile constraint matching                     │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│        LanceDB Vector Store (vector_store.py)           │
│  ┌───────────────────────────────────────────────────┐  │
│  │ Table: method_cards                               │  │
│  │ • content: embedding text                         │  │
│  │ • vector: 1536-dim embedding                      │  │
│  │ • metadata: {topic, category, constraints}        │  │
│  │ • card_json: full method card                     │  │
│  └───────────────────────────────────────────────────┘  │
│                                                         │
│  Two-stage retrieval:                                   │
│  1. Vector similarity search (semantic)                 │
│  2. Reranking (keyword + topic boosting)                │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│         Constraint-Based Filtering                      │
│  • Check data profile matches card requirements         │
│  • Score applicability (0.0 - 1.0)                      │
│  • Return ranked results                                │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│              Top-K Method Cards                         │
│  1. SimpleImputer (median) - score: 0.95               │
│  2. SimpleImputer (mean) - score: 0.70                 │
│  3. KNNImputer - score: 0.60                           │
└─────────────────────────────────────────────────────────┘
```

## 3. Setup and Imports

Let's start with the necessary imports and initialization.

In [1]:
import sys
from pathlib import Path
import json
import yaml
from pprint import pprint

# Add parent directory to path AFTER current directory to avoid shadowing
# Remove current directory from beginning of sys.path if present
if '' in sys.path:
    sys.path.remove('')
if str(Path.cwd()) in sys.path:
    sys.path.remove(str(Path.cwd()))

# Add parent directory
parent_path = str(Path.cwd().parent)
if parent_path not in sys.path:
    sys.path.append(parent_path)

# Import Agentic Analytics components
from src.rag.rag_system import RAGSystem
from src.rag.method_card import MethodCard, MethodCategory, ProblemType, DataConditions
from src.config import config

print(" Imports successful!")
print(f" Working directory: {Path.cwd()}")
print(f" Vector store type: {config.vector_store_type}")
print(f" LLM provider: {config.llm_provider}")

# Note: If LanceDB connection fails, the system will automatically fall back to FAISS
print("\n Note: This notebook demonstrates the method card system.")
print("   If you see LanceDB errors, ensure method cards are loaded first:")
print("   Run: python testing/load_method_cards.py")

 Imports successful!
 Working directory: /Users/luofangyu/git/AgenticAnalytics/examples
 Vector store type: lancedb
 LLM provider: openai

 Note: This notebook demonstrates the method card system.
   If you see LanceDB errors, ensure method cards are loaded first:
   Run: python testing/load_method_cards.py


## 4. Method Card Structure Deep Dive

Let's examine a real method card from our knowledge base.

In [2]:
# Load and display a real method card from YAML
method_cards_dir = Path.cwd().parent / "method_cards"
imputation_file = method_cards_dir / "imputation.yaml"

with open(imputation_file, 'r') as f:
    imputation_cards = yaml.safe_load(f)

# Display the first method card (SimpleImputer median)
median_imputer = imputation_cards[1]  # Index 1 is median

median_imputer


{'method_name': 'SimpleImputer (median)',
 'category': 'preprocessing_imputation',
 'problem_type': 'missing_values',
 'data_conditions': {'sample_size_min': 10,
  'sample_size_max': None,
  'sample_size_recommended': 100,
  'normality_required': False,
  'handles_non_normal': True,
  'handles_missing': True,
  'requires_complete_data': False,
  'handles_categorical': False,
  'requires_numeric': True,
  'supports_binary_target': False,
  'supports_multiclass_target': False,
  'supports_continuous_target': False,
  'requires_independence': False,
  'handles_outliers': True,
  'requires_balanced_classes': False},
 'assumptions': ['Missing data is MCAR or MAR', 'Data is numeric'],
 'when_to_use': 'Numeric features with missing values. Skewed distributions. Outliers present. Missing percentage < 30%. More robust than mean for non-normal data.',
 'when_not_to_use': 'Categorical features (use mode). Missing > 40%. Time series with trend. Need to preserve mean (use mean imputer).',
 'alterna

## 5. How Method Cards are Indexed

The indexing process transforms YAML method cards into searchable vectors with metadata.

In [3]:
# Demonstrate the indexing process

# Create a MethodCard object from the YAML data
card_data = median_imputer.copy()
card_data['category'] = MethodCategory(card_data['category'])
card_data['problem_type'] = ProblemType(card_data['problem_type'])
card_data['data_conditions'] = DataConditions(**card_data['data_conditions'])

card = MethodCard(**card_data)

print("="*70)
print("INDEXING PROCESS")
print("="*70)

# Step 1: Convert to embedding text (semantic representation)
embedding_text = card.to_embedding_text()
print(f"\n Embedding Text (first 500 chars):")
print(f"{embedding_text[:500]}...")

# Step 2: Extract metadata for filtering
metadata = card.to_metadata()
print(f"\n Metadata for Filtering:")
pprint(metadata)

# Step 3: Full card stored as JSON for reconstruction
card_json = json.dumps(card.to_dict(), indent=2)
print(f"\n Full Card JSON (first 300 chars):")
print(f"{card_json[:300]}...")

print(f"\n💡 Key Points:")
print(f"   • Embedding text is rich semantic representation")
print(f"   • Metadata enables fast filtering (topic, category, constraints)")
print(f"   • Full card JSON allows complete reconstruction")
print(f"   • Vector embedding (1536-dim) computed from embedding text")

INDEXING PROCESS

 Embedding Text (first 500 chars):
Method: SimpleImputer (median) | Category: preprocessing_imputation | Problem: missing_values | When to use: Numeric features with missing values. Skewed distributions. Outliers present. Missing percentage < 30%. More robust than mean for non-normal data. | Assumptions: Missing data is MCAR or MAR, Data is numeric | When NOT to use: Categorical features (use mode). Missing > 40%. Time series with trend. Need to preserve mean (use mean imputer). | Alternatives: SimpleImputer (mean), SimpleImputer...

 Metadata for Filtering:
{'category': 'preprocessing_imputation',
 'doc_type': 'method_card',
 'handles_categorical': False,
 'handles_missing_values': True,
 'method_name': 'SimpleImputer (median)',
 'problem_type': 'missing_values',
 'requires_normality': False,
 'sample_size_max': 999999,
 'sample_size_min': 10,
 'source': 'sklearn',
 'source_license': 'BSD-3-Clause',
 'source_url': 'https://scikit-learn.org/stable/modules/generated/sk

## 6. Basic Retrieval Examples

Now let's see the RAG system in action with different queries.

In [4]:
# Check if method cards are loaded
lancedb_path = Path.cwd().parent / "lancedb"
method_cards_table = lancedb_path / "method_cards.lance"

if method_cards_table.exists():
    print("Method cards table found!")
    print(f"Location: {method_cards_table}")
else:
    print(" Method cards not yet loaded.")

Method cards table found!
Location: /Users/luofangyu/git/AgenticAnalytics/lancedb/method_cards.lance


In [5]:
# Set the correct LanceDB path
lancedb_path = str(Path.cwd().parent / "lancedb")

# Patch RAGSystem to always use the correct db_path for LanceDBVectorStore
import src.rag.vector_store as vector_store_mod
orig_lancedb_vs_init = vector_store_mod.LanceDBVectorStore.__init__

def patched_init(self, embeddings=None, db_path=None, table_name="analytics_rag"):
    if db_path is None:
        db_path = lancedb_path
    orig_lancedb_vs_init(self, embeddings=embeddings, db_path=db_path, table_name=table_name)

vector_store_mod.LanceDBVectorStore.__init__ = patched_init


In [6]:
# Initialize RAG system
rag = RAGSystem()

# Test queries across different categories
test_queries = [
    ("correlation between variables", "statistics"),
    ("how to handle missing values", "preprocessing"),
    ("test if data is normally distributed", "statistics"),
    ("binary classification model", "modeling"),
    ("scale features for machine learning", "preprocessing"),
    
]

print("="*70)
print("BASIC RETRIEVAL EXAMPLES (No Data Profile)")
print("="*70)

for query, expected_topic in test_queries:
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print(f"Expected Topic: {expected_topic}")
    print(f"{'='*70}")
    
    try:
        results = rag.retrieve_method_cards(query, k=3)
        
        if results:
            print(f"\n Found {len(results)} methods:")
            for i, (card, score) in enumerate(results, 1):
                print(f"\n   {i}. {card.method_name}" + f"   ||   Category: {card.category.value}")
                print(f"      Confidence: {score:.2f}")
                print(f"      When to use: {card.when_to_use[:120]}...")
        else:
            print(f"❌ No results found")
            
    except Exception as e:
        print(f"❌ Error: {e}")

[LanceDB] Opened existing table: analytics_rag
BASIC RETRIEVAL EXAMPLES (No Data Profile)

Query: 'correlation between variables'
Expected Topic: statistics
[LanceDB] Opened existing table: method_cards

 Found 3 methods:

   1. Pearson Correlation   ||   Category: stats_correlation
      Confidence: 1.00
      When to use: Measure linear correlation between two continuous variables. Both variables approximately normal. Looking for monotonic ...

   2. Spearman Correlation   ||   Category: stats_correlation
      Confidence: 1.00
      When to use: Measure monotonic (not necessarily linear) correlation. Non-normal data or ordinal data. Outliers present. Small samples...

   3. Chi-Square Test of Independence   ||   Category: stats
      Confidence: 1.00
      When to use: Use chi-square test to examine association between two categorical variables. Tests if the distribution of one variable ...

Query: 'how to handle missing values'
Expected Topic: preprocessing
[LanceDB] Opened existin

## 8. Constraint-Based Retrieval

The real power of method cards comes from **constraint-based filtering** - matching methods to actual data characteristics.

In [7]:
# Example 1: Small dataset with missing values and outliers
print("="*70)
print("CONSTRAINT-BASED RETRIEVAL EXAMPLE 1")
print("="*70)

small_dataset_profile = {
    "shape": {"rows": 100, "columns": 5},
    "missing_values": {
        "has_missing": True,
        "columns_with_missing": ["age", "income"]
    },
    "has_non_normal": True,
    "has_outliers": True,
    "categorical_columns": []
}

print(f"🔍 Query: 'impute missing values'")
print(f"{'='*70}")

results = rag.retrieve_method_cards(
    "impute missing values",
    data_profile=small_dataset_profile,
    k=5
)

if results:
    print(f"Found {len(results)} applicable methods:")
    for i, (card, applicability_score) in enumerate(results, 1):
        # Check constraint matching
        passes_constraints, _ = card.matches_data_profile(small_dataset_profile)
        
        print(f"{i}. {card.method_name}" )
        print(f"    Applicability Score: {applicability_score:.2f}" + f"  ||  Passes Hard Constraints: {passes_constraints}")
        
        # Show why it's suitable
        dc = card.data_conditions
        print(f"    Data Requirements:")
        print(f"      • Min samples: {dc.sample_size_min} (we have: 100) ✓")
        print(f"      • Handles missing: {dc.handles_missing_values} ✓")
        print(f"      • Handles outliers: {dc.handles_outliers} {'✓' if dc.handles_outliers else '✗'}")
        print(f"      • Handles non-normal: {dc.handles_non_normal} {'✓' if dc.handles_non_normal else '✗'}")
        print()

CONSTRAINT-BASED RETRIEVAL EXAMPLE 1

 Data Profile:
   • Rows: 100
   • Columns: 5
   • Missing values: Yes (age, income)
   • Distribution: Non-normal
   • Outliers: Yes

🔍 Query: 'impute missing values'
[LanceDB] Opened existing table: method_cards

 Found 3 applicable methods:

1. SimpleImputer (mean)
    Applicability Score: 1.00
    Passes Hard Constraints: True
    Data Requirements:
      • Min samples: 10 (we have: 100) ✓
      • Handles missing: True ✓
      • Handles outliers: False ✗
      • Handles non-normal: True ✓

2. SimpleImputer (median)
    Applicability Score: 1.00
    Passes Hard Constraints: True
    Data Requirements:
      • Min samples: 10 (we have: 100) ✓
      • Handles missing: True ✓
      • Handles outliers: True ✓
      • Handles non-normal: True ✓

3. SimpleImputer (mode)
    Applicability Score: 1.00
    Passes Hard Constraints: True
    Data Requirements:
      • Min samples: 10 (we have: 100) ✓
      • Handles missing: True ✓
      • Handles outliers

In [8]:
# Example 2: Large dataset for modeling
print("="*70)
print("CONSTRAINT-BASED RETRIEVAL EXAMPLE 2")
print("="*70)

large_dataset_profile = {
    "shape": {"rows": 50000, "columns": 20},
    "missing_values": {"has_missing": False},
    "has_non_normal": True,
    "has_outliers": False,
    "categorical_columns": ["gender", "region"]
}

print(f"\n🔍 Query: 'classification model'")
print(f"{'='*70}")

results = rag.retrieve_method_cards(
    "classification model",
    data_profile=large_dataset_profile,
    k=5
)

if results:
    print(f"\n✅ Found {len(results)} applicable methods:\n")
    for i, (card, applicability_score) in enumerate(results, 1):
        passes_constraints, _ = card.matches_data_profile(large_dataset_profile)
        
        print(f"{i}. {card.method_name}")
        print(f"   📈 Applicability Score: {applicability_score:.2f}")
        print(f"   ✓ Passes Constraints: {passes_constraints}")
        
        # Show constraint details
        dc = card.data_conditions
        sample_ok = (dc.sample_size_min or 0) <= 50000 <= (dc.sample_size_max or float('inf'))
        print(f"   📋 Sample size: {dc.sample_size_min}-{dc.sample_size_max or '∞'} {'✓' if sample_ok else '✗'}")
        print(f"   📋 Handles non-normal: {dc.handles_non_normal} {'✓' if dc.handles_non_normal else '✗'}")
        print()

CONSTRAINT-BASED RETRIEVAL EXAMPLE 2

📊 Data Profile:
   • Rows: 50,000
   • Columns: 20
   • Missing values: No
   • Distribution: Non-normal
   • Categorical features: 2

🔍 Query: 'classification model'
[LanceDB] Opened existing table: method_cards

✅ Found 5 applicable methods:

1. Logistic Regression
   📈 Applicability Score: 0.40
   ✓ Passes Constraints: False
   📋 Sample size: 30-∞ ✓
   📋 Handles non-normal: None ✗

2. LogisticRegression
   📈 Applicability Score: 0.40
   ✓ Passes Constraints: False
   📋 Sample size: 50-∞ ✓
   📋 Handles non-normal: True ✓

3. RandomForestClassifier
   📈 Applicability Score: 0.40
   ✓ Passes Constraints: False
   📋 Sample size: 100-∞ ✓
   📋 Handles non-normal: True ✓

4. GradientBoostingClassifier
   📈 Applicability Score: 0.40
   ✓ Passes Constraints: False
   📋 Sample size: 200-100000 ✓
   📋 Handles non-normal: True ✓

5. Lasso Regression (L1)
   📈 Applicability Score: 0.40
   ✓ Passes Constraints: False
   📋 Sample size: 20-∞ ✓
   📋 Handles non-

## 9. Category-Specific Retrieval

The RAG system provides specialized retrieval methods for different domains.

In [9]:
# Test preprocessing methods only
print(f"\n{'='*70}")
print(f" PREPROCESSING METHODS")
print(f"{'='*70}")
print(f" Query: 'handle missing data'\n")

preprocessing_results = rag.retrieve_methods_for_preprocessing("handle missing data", k=3)
for i, (card, score) in enumerate(preprocessing_results, 1):
    print(f"{i}. {card.method_name}")
    print(f"   Category: {card.category.value}")
    print(f"   Library: {card.library}")

# Test statistical tests only
print(f"\n{'='*70}")
print(f" STATISTICAL TESTS")
print(f"{'='*70}")
print(f" Query: 'test normality'\n")


stats_results = rag.retrieve_methods_for_statistics("test normality", k=3)
for i, (card, score) in enumerate(stats_results, 1):
    print(f"{i}. {card.method_name}")
    print(f"   Category: {card.category.value}")
    print(f"   Sample size range: {card.data_conditions.sample_size_min}-{card.data_conditions.sample_size_max or '∞'}")



 PREPROCESSING METHODS
 Query: 'handle missing data'

[LanceDB] Opened existing table: method_cards
1. SimpleImputer (median)
   Category: preprocessing_imputation
   Library: sklearn
2. SimpleImputer (mean)
   Category: preprocessing_imputation
   Library: sklearn
3. SimpleImputer (mode)
   Category: preprocessing_imputation
   Library: sklearn

 STATISTICAL TESTS
 Query: 'test normality'

[LanceDB] Opened existing table: method_cards
1. SimpleImputer (median)
   Category: preprocessing_imputation
   Library: sklearn
2. SimpleImputer (mean)
   Category: preprocessing_imputation
   Library: sklearn
3. SimpleImputer (mode)
   Category: preprocessing_imputation
   Library: sklearn

 STATISTICAL TESTS
 Query: 'test normality'

[LanceDB] Opened existing table: method_cards
1. Shapiro-Wilk
   Category: stats_normality
   Sample size range: 3-5000
2. Anderson-Darling
   Category: stats_normality
   Sample size range: 8-∞
3. Kolmogorov-Smirnov
   Category: stats_normality
   Sample size rang

In [10]:
print("="*70)
print("CATEGORY-SPECIFIC RETRIEVAL")
print("="*70)

# Test models only
print(f"\n{'='*70}")
print(f" MACHINE LEARNING MODELS")
print(f"{'='*70}")
print(f"🔍 Query: 'binary classification'\n")

model_results = rag.retrieve_methods_for_modeling("binary classification", k=3)
for i, (card, score) in enumerate(model_results, 1):
    print(f"{i}. {card.method_name}")
    print(f"   Category: {card.category.value}")
    print(f"   Min samples: {card.data_conditions.sample_size_min}")
    print(f"   When to use: {card.when_to_use[:80]}...")

CATEGORY-SPECIFIC RETRIEVAL

 MACHINE LEARNING MODELS
🔍 Query: 'binary classification'

[LanceDB] Opened existing table: method_cards
1. Logistic Regression
   Category: model
   Min samples: 30
   When to use: Use Logistic Regression for binary classification when you need probabilistic pr...
2. LogisticRegression
   Category: model_classification
   Min samples: 50
   When to use: Good baseline for binary or multiclass classification. Fast training and predict...
3. GradientBoostingClassifier
   Category: model_classification
   Min samples: 200
   When to use: High accuracy needed. Non-linear patterns. Feature interactions. Willing to tune...



### Resources

- **Method Cards Directory**: `method_cards/`
- **Loading Script**: `testing/load_method_cards.py`
- **Testing Examples**: `testing/test_method_card_retrieval.py`
- **RAG System**: `src/rag/rag_system.py`
- **Documentation**: `docs/METHOD_CARDS.md`